In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
import json

In [ ]:
# import getpass
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "false").lower() == "true"
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY", None)

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "kimi-k2:1t-cloud")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")
OLLAMA_MODEL_URL = os.getenv("OLLAMA_MODEL_URL", "http://localhost:11434")

print(f"LANGSMITH_TRACING: {LANGSMITH_TRACING}")
print(f"LANGSMITH_API_KEY: {LANGSMITH_API_KEY or 'not set'}")
print(f"OLLAMA_MODEL: {OLLAMA_MODEL}")
print(f"OLLAMA_API_KEY: {OLLAMA_API_KEY or 'not set'}")
print(f"OLLAMA_MODEL_URL: {OLLAMA_MODEL_URL}")

args = {
  "model": OLLAMA_MODEL,
  "model_provider": "openai",
  "api_key": OLLAMA_API_KEY,
  "base_url": f"{OLLAMA_MODEL_URL}/v1",
}

model = init_chat_model(
  args["model"],
  model_provider=args["model_provider"],
  api_key=args["api_key"],
  base_url=args["base_url"],
)

print(json.dumps(args, indent=2))

# 1. Define custom state
First, define a custom state schema that tracks which step is currently active:

The current_step field is the core of the state machine pattern - it determines which configuration (prompt + tools) is loaded on each turn.

In [ ]:
from langchain.agents import AgentState
from typing_extensions import NotRequired
from typing import Literal

# Define the possible workflow steps
SupportState = Literal["warranty_collector", "issue_classifier", "resolution_specialist"] 

class SupportState(AgentState):
    """State for customer support workflow."""
    current_step: NotRequired[SupportState]
    warranty_status: NotRequired[Literal["in_warranty", "out_of_warranty"]]
    issue_type: NotRequired[Literal["hardware", "software"]]

# 2. Create tools that manage workflow state

Create tools that update the workflow state. These tools allow the agent to record information and transition to the next step.

The key is using `Command` to update state, including the `current_step` field:

Notice how `record_warranty_status` and `record_issue_type` return Command objects that update both the data (`warranty_status`, `issue_type`) AND the `current_step`. This is how the state machine works - tools control workflow progression.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def record_warranty_status(
    status: Literal["in_warranty", "out_of_warranty"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:  
    """Record the customer's warranty status and transition to issue classification."""
    return Command(  
        update={  
            "messages": [
                ToolMessage(
                    content=f"Warranty status recorded as: {status}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "warranty_status": status,
            "current_step": "issue_classifier",  
        }
    )

@tool
def record_issue_type(
    issue_type: Literal["hardware", "software"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:  
    """Record the type of issue and transition to resolution specialist."""
    return Command(  
        update={  
            "messages": [
                ToolMessage(
                    content=f"Issue type recorded as: {issue_type}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "issue_type": issue_type,
            "current_step": "resolution_specialist",  
        }
    )

@tool
def escalate_to_human(reason: str) -> str:
    """Escalate the case to a human support specialist."""
    # In a real system, this would create a ticket, notify staff, etc.
    return f"Escalating to human support. Reason: {reason}"

@tool
def provide_solution(solution: str) -> str:
    """Provide a solution to the customer's issue."""
    return f"Solution provided: {solution}"

# 3. Define step configurations

Define prompts and tools for each step. First, define the prompts for each step:

In [ ]:
# Define prompts as constants for easy reference
WARRANTY_COLLECTOR_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Warranty verification

At this step, you need to:
1. Greet the customer warmly
2. Ask if their device is under warranty
3. Use record_warranty_status to record their response and move to the next step

Be conversational and friendly. Don't ask multiple questions at once."""

ISSUE_CLASSIFIER_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Issue classification
CUSTOMER INFO: Warranty status is {warranty_status}

At this step, you need to:
1. Ask the customer to describe their issue
2. Determine if it's a hardware issue (physical damage, broken parts) or software issue (app crashes, performance)
3. Use record_issue_type to record the classification and move to the next step

If unclear, ask clarifying questions before classifying."""

RESOLUTION_SPECIALIST_PROMPT = """You are a customer support agent helping with device issues.

CURRENT STAGE: Resolution
CUSTOMER INFO: Warranty status is {warranty_status}, issue type is {issue_type}

At this step, you need to:
1. For SOFTWARE issues: provide troubleshooting steps using provide_solution
2. For HARDWARE issues:
   - If IN WARRANTY: explain warranty repair process using provide_solution
   - If OUT OF WARRANTY: escalate_to_human for paid repair options

Be specific and helpful in your solutions."""

Then map step names to their configurations using a dictionary:

In [ ]:
# Step configuration: maps step name to (prompt, tools, required_state)
STEP_CONFIG = {
    "warranty_collector": {
        "prompt": WARRANTY_COLLECTOR_PROMPT,
        "tools": [record_warranty_status],
        "requires": [],
        "icon": "🛡️",
    },
    "issue_classifier": {
        "prompt": ISSUE_CLASSIFIER_PROMPT,
        "tools": [record_issue_type],
        "requires": ["warranty_status"],
        "icon": "🔍",
    },
    "resolution_specialist": {
        "prompt": RESOLUTION_SPECIALIST_PROMPT,
        "tools": [provide_solution, escalate_to_human],
        "requires": ["warranty_status", "issue_type"],
        "icon": "🛠️",
    },
}

This dictionary-based configuration makes it easy to:
* See all steps at a glance
* Add new steps (just add another entry)
* Understand the workflow dependencies (`requires` field)
* Use prompt templates with state variables (e.g., `{warranty_status}`)

# 4. Create step-based middleware

Create middleware that reads `current_step` from state and applies the appropriate configuration. We’ll use the `@wrap_model_call` decorator for a clean implementation:

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """Configure agent behavior based on the current step."""
    # Get current step (defaults to warranty_collector for first interaction)
    current_step = request.state.get("current_step", "warranty_collector")  

    # Look up step configuration
    stage_config = STEP_CONFIG[current_step]  

    # Validate required state exists
    for key in stage_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"{key} must be set before reaching {current_step}")

    # Format prompt with state values (supports {warranty_status}, {issue_type}, etc.)
    system_prompt = stage_config["prompt"].format(**request.state)

    # Inject system prompt and step-specific tools
    request = request.override(  
        system_prompt=system_prompt,  
        tools=stage_config["tools"],  
    )

    return handler(request)

This middleware:
1. Reads current step: Gets current_step from state (defaults to “warranty_collector”).
2. Looks up configuration: Finds the matching entry in STEP_CONFIG.
3. Validates dependencies: Ensures required state fields exist.
4. Formats prompt: Injects state values into the prompt template.
5. Applies configuration: Overrides the system prompt and available tools.

The `request.override()` method is key - it allows us to dynamically change the agent’s behavior based on state without creating separate agent instances.

# 5. Create the agent

Now create the agent with the step-based middleware and a checkpointer for state persistence:

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Collect all tools from all step configurations
all_tools = [
    record_warranty_status,
    record_issue_type,
    provide_solution,
    escalate_to_human,
]

# Create the agent with step-based configuration
agent = create_agent(
    model,
    tools=all_tools,
    state_schema=SupportState,  
    middleware=[apply_step_config],  
    checkpointer=InMemorySaver(),  
)

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

# 6. Test the workflow

Test the complete workflow:

In [ ]:
from langchain.messages import HumanMessage
import uuid

# Start a new conversation thread
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print("Customer Support Chat (type 'quit' to exit)")
print("=" * 50)

while True:
    user_input = input("\nYou: ").strip()
    if not user_input or user_input.lower() == "quit":
        print("Chat ended.")
        break

    result = agent.invoke(
        {"messages": [HumanMessage(user_input)]},
        config,
    )

    # Print only the last AI message
    for msg in result["messages"][::-1]:
        if msg.type == "ai" and msg.content:
            print(f"\nAgent: {msg.content}")
            break

    step = result.get("current_step", "warranty_collector")
    icon = STEP_CONFIG.get(step, {}).get("icon", "")
    print(f"[Step: {icon} {step}]")